# Liver Disease Diagnosis

End-to-end notebook for preprocessing, SMOTE balancing, and model training.

In [1]:
# If running in Colab, install dependencies.
# You can comment this out if packages are already available.
!pip -q install miceforest imbalanced-learn xgboost

In [2]:
import pandas as pd
import numpy as np
from miceforest import ImputationKernel
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

FEATURE_COLUMNS = [
    'Age', 'Gender', 'Total Bilirubin', 'Direct Bilirubin',
    'Alkphos Alkaline Phosphotase', 'Sgpt Alamine Aminotransferase',
    'Sgot Aspartate Aminotransferase', 'Total Protiens',
    'ALB Albumin', 'A/G Ratio Albumin and Globulin Ratio'
]
TARGET_COLUMN = 'Diagnosis'
SCALING_COLUMNS = [
    'Alkphos Alkaline Phosphotase',
    'Sgpt Alamine Aminotransferase',
    'Sgot Aspartate Aminotransferase'
]

In [3]:
def impute_data(data: pd.DataFrame) -> pd.DataFrame:
    print('Imputing missing values...')
    imputer = ImputationKernel(data, random_state=42, save_all_iterations_data=True)
    imputer.mice(5)
    out = imputer.complete_data()
    print('Missing values imputed successfully.')
    return out

def apply_smote(data: pd.DataFrame):
    print('Applying SMOTE for class balancing...')
    sm = SMOTE(random_state=42)
    X_resampled, y_resampled = sm.fit_resample(data[FEATURE_COLUMNS], data[TARGET_COLUMN])
    print('SMOTE applied successfully.')
    return X_resampled, y_resampled

def normalize_and_scale_features(data: pd.DataFrame) -> pd.DataFrame:
    print('Normalizing and scaling specific features...')
    for col in SCALING_COLUMNS:
        col_range = data[col].max() - data[col].min()
        if col_range == 0:
            data[col] = 0
        else:
            data[col] = (data[col] - data[col].min()) / col_range * 100
    print('Normalization and scaling completed.')
    return data

def load_data(filename='/content/LD_raw_data.csv') -> pd.DataFrame:
    data = pd.read_csv(filename)
    data = impute_data(data)
    X_resampled, y_resampled = apply_smote(data)
    data = pd.DataFrame(X_resampled, columns=FEATURE_COLUMNS)
    data[TARGET_COLUMN] = y_resampled
    data = normalize_and_scale_features(data)
    return data

In [5]:
# Upload LD_raw_data.csv to /content first, or update the path below.
DATA_PATH = '/content/LD_raw_data.csv'

data = load_data(DATA_PATH)
X, y = data.drop(TARGET_COLUMN, axis=1), data[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train/Test shapes:', X_train.shape, X_test.shape)

Imputing missing values...
Missing values imputed successfully.
Applying SMOTE for class balancing...
SMOTE applied successfully.
Normalizing and scaling specific features...
Normalization and scaling completed.
Train/Test shapes: (18819, 10) (4705, 10)


In [7]:
def save_grid_search_results(filename, grid_search):
    with open(filename, 'w') as f:
        f.write(f'Best score: {grid_search.best_score_}\n')
        f.write(f'Best parameters: {grid_search.best_params_}\n')

def evaluate_classifier(estimator, param_grid, X_train, y_train, X_test, y_test, filename, cv=3):
    grid_search = GridSearchCV(
        estimator, param_grid, scoring='accuracy', cv=cv, n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)
    y_pred = grid_search.best_estimator_.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'Accuracy of {type(estimator).__name__}: {acc:.4f}')
    save_grid_search_results(filename, grid_search)

# Set FAST_MODE=False to use larger grids (slower).
FAST_MODE = True

if FAST_MODE:
    rf_params = {'n_estimators': [100], 'max_depth': [5, 10], 'min_samples_split': [2], 'min_samples_leaf': [1], 'criterion': ['gini'], 'max_features': ['sqrt'], 'bootstrap': [True], 'random_state': [0]}
    dt_params = {'max_depth': [5, 10], 'criterion': ['gini', 'entropy'], 'splitter': ['best'], 'min_samples_split': [2, 10], 'max_features': ['sqrt', None], 'class_weight': [None], 'random_state': [0]}
    knn_params = {'n_neighbors': [3, 5, 7], 'p': [1, 2], 'metric': ['minkowski'], 'weights': ['uniform', 'distance']}
    lr_params = {'C': [0.1, 1, 10], 'solver': ['lbfgs', 'liblinear'], 'max_iter': [500], 'random_state': [0]}
    svc_params = {'C': [1, 10], 'kernel': ['linear', 'rbf'], 'gamma': [0.1, 1], 'random_state': [0]}
    xgb_params = {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5], 'random_state': [0]}
else:
    rf_params = {'n_estimators': [10, 100, 1000], 'max_depth': list(range(1, 11)), 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 5, 10], 'criterion': ['gini', 'entropy'], 'max_features': ['sqrt', 'log2', None], 'bootstrap': [True, False], 'random_state': [0]}
    dt_params = {'max_depth': list(range(1, 11)), 'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'], 'min_samples_split': [2, 5, 10, 20], 'max_features': ['sqrt', 'log2', None], 'class_weight': ['balanced', None], 'random_state': [0]}
    knn_params = {'n_neighbors': list(range(1, 11)), 'p': [1, 2], 'metric': ['minkowski', 'euclidean', 'manhattan'], 'weights': ['uniform', 'distance']}
    lr_params = {'C': [0.1, 1, 10, 100, 1000], 'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'], 'max_iter': [200, 500, 1000], 'random_state': [0]}
    svc_params = {'C': [0.1, 1, 10, 100, 1000], 'kernel': ['linear', 'poly', 'rbf', 'sigmoid'], 'gamma': [0.1, 1, 10, 100], 'random_state': [0]}
    xgb_params = {'n_estimators': [100, 200, 300], 'learning_rate': [0.1, 0.2, 0.3], 'max_depth': [1, 2, 3, 4, 5, 6], 'random_state': [0]}

In [8]:
jobs = [
    (RandomForestClassifier(), rf_params, 'RF_results.txt'),
    (DecisionTreeClassifier(), dt_params, 'DT_results.txt'),
    (KNeighborsClassifier(), knn_params, 'KNN_results.txt'),
    (LogisticRegression(), lr_params, 'LR_results.txt'),
    (SVC(), svc_params, 'SVC_results.txt'),
    (XGBClassifier(), xgb_params, 'XGB_results.txt'),
]

for estimator, params, output_file in jobs:
    print(f'\nTraining {type(estimator).__name__} ...')
    evaluate_classifier(estimator, params, X_train, y_train, X_test, y_test, output_file, cv=3)

print('\nDone. Result files saved in current working directory.')


Training RandomForestClassifier ...
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Accuracy of RandomForestClassifier: 0.9265

Training DecisionTreeClassifier ...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
Accuracy of DecisionTreeClassifier: 0.9022

Training KNeighborsClassifier ...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Accuracy of KNeighborsClassifier: 0.9088

Training LogisticRegression ...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Accuracy of LogisticRegression: 0.7046

Training SVC ...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy of SVC: 0.8519

Training XGBClassifier ...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy of XGBClassifier: 0.9996

Done. Result files saved in current working directory.


In [9]:
# Optional: download all result files from Colab.
import glob
import zipfile
from google.colab import files

result_files = glob.glob('*_results.txt')
if result_files:
    with zipfile.ZipFile('model_results.zip', 'w') as zf:
        for file in result_files:
            zf.write(file)
    files.download('model_results.zip')
else:
    print('No *_results.txt files found yet.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

# Retrain best model (example: Random Forest)
best_grid = GridSearchCV(
    RandomForestClassifier(), rf_params,
    scoring='accuracy', cv=3, n_jobs=-1, verbose=1
)
best_grid.fit(X_train, y_train)
best_model = best_grid.best_estimator_

# Test on held-out data
y_pred = best_model.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

if hasattr(best_model, "predict_proba"):
  y_prob = best_model.predict_proba(X_test)[:, 1]
  print("ROC-AUC:", roc_auc_score(y_test, y_prob))

joblib.dump(best_model, "best_liver_model.pkl")
print("Saved best_liver_model.pkl")

Fitting 3 folds for each of 2 candidates, totalling 6 fits
Confusion matrix:
[[2351    4]
 [ 342 2008]]

Classification report:
              precision    recall  f1-score   support

           0     0.8730    0.9983    0.9315      2355
           1     0.9980    0.8545    0.9207      2350

    accuracy                         0.9265      4705
   macro avg     0.9355    0.9264    0.9261      4705
weighted avg     0.9354    0.9265    0.9261      4705

ROC-AUC: 0.9961888241405792
Saved best_liver_model.pkl


In [12]:
sample = X_test.head(10)
sample_true = y_test.head(10)
sample_pred = best_model.predict(sample)

check = pd.DataFrame({
    'true': sample_true.values,
    'pred': sample_pred
})
print(check)

   true  pred
0     1     1
1     1     1
2     0     0
3     0     0
4     0     0
5     0     0
6     1     1
7     1     1
8     1     1
9     1     0


In [13]:
new_patient = pd.DataFrame([{
    'Age': 45,
    'Gender': 1,
    'Total Bilirubin': 1.2,
    'Direct Bilirubin': 0.4,
    'Alkphos Alkaline Phosphotase': 85,
    'Sgpt Alamine Aminotransferase': 40,
    'Sgot Aspartate Aminotransferase': 35,
    'Total Protiens': 7.0,
    'ALB Albumin': 4.0,
    'A/G Ratio Albumin and Globulin Ratio': 1.2
}])

prediction = best_model.predict(new_patient)[0]
print("Predicted Diagnosis:", prediction)  # 0 or 1 depending on your dataset

if hasattr(best_model, "predict_proba"):
    prob = best_model.predict_proba(new_patient)[0]
    print("Probabilities:", prob)

Predicted Diagnosis: 1
Probabilities: [0.04590515 0.95409485]
